In [1]:
import typing as T
import pickle
import pathlib as P
import itertools as it

In [2]:
import numpy as np
import pandas as pd
import sklearn.preprocessing as prep
from tqdm import tqdm
import scipy.sparse as sp

In [3]:
prj_root = P.Path("__file__").absolute().parent.parent.parent
import sys
if str(prj_root) not in sys.path:
  sys.path.append(str(prj_root))

import util.metrics as hm
import sklearn.preprocessing as skp
import util.obo_parser as gp

In [4]:
ns = ["cc", "mf", "bp"]
ontology_lst = ["cellular_component", "molecular_function", "biological_process"]

In [5]:
methodpred_path = [P.Path(f"/data0/shaojiangyi/dpfunc/netgo-results/DPFunc_{x}_final.pkl") for x in ns]

In [6]:
def sigfunc(x):
    return 1/(1+np.exp(-x))

In [7]:
def load_pred(predpath: P.Path,):
    """
    Load the sprofgo prediction file.
    """
    with open(predpath, "rb") as f:
        ori_pred = pickle.load(f)

    assert hasattr(ori_pred, "predictions"), "The loaded object does not have 'predictions' attribute."
    pred_col = ori_pred.predictions
    term_set = set(it.chain.from_iterable(ori_pred.predictions.apply(lambda x: list(x.keys())).to_list()))
    term_idx = {k: i for i, k in enumerate(term_set)}
    preds = np.zeros((len(pred_col), len(term_set)), dtype=np.float32)
    for i, row in enumerate(pred_col):
        for term, score in row.items():
            preds[i, term_idx[term]] = score
    prot_names = ori_pred.protein_id
    go_terms = list(term_set)

    # mean = np.mean(preds, axis=0)
    # stds = np.std(preds, axis=0)
    # preds = sigfunc((preds - mean) / stds)
    return prot_names, go_terms, preds

In [8]:
methodpreds = [load_pred(p) for p in methodpred_path]

In [9]:
# del methodpreds

In [10]:
nspace_path = P.Path("/data0/shaojiangyi/data/data-netgo/namespace_terms.pkl")
with open(nspace_path, "rb") as h:
   namespace_terms = pickle.load(h)

In [11]:
terms_lst = [namespace_terms[x] for x in ontology_lst]

In [12]:
def align_labels(predicted_results, o1, o2, fill_value=0.0):
    """
    Align predicted results to match new label order, handling missing labels.
    
    Args:
        predicted_results: numpy array of shape (N, len(o1))
        o1: list - original label order
        o2: list - desired label order (can have different labels/length)
        fill_value: value to use for labels in o2 that don't exist in o1
    
    Returns:
        numpy array of shape (N, len(o2)) - aligned predictions
    """
    N = predicted_results.shape[0]
    M_new = len(o2)
    
    # Initialize new result array
    aligned_results = np.full((N, M_new), fill_value, dtype=predicted_results.dtype)
    
    # Create mapping from label to column index in o1
    o1_label_to_idx = {label: idx for idx, label in enumerate(o1)}
    
    # Fill the aligned results
    for new_idx, label in enumerate(o2):
        if label in o1_label_to_idx:
            old_idx = o1_label_to_idx[label]
            aligned_results[:, new_idx] = predicted_results[:, old_idx]
        # If label not in o1, keep the fill_value (already set)
    
    return aligned_results

In [13]:
methodaligned = [
    align_labels(tp[2], tp[1] ,o2)
    for tp, o2 in zip(methodpreds, terms_lst)
]

In [14]:
methodaligned[0].shape

(259, 2880)

In [15]:
dataset_path = P.Path("/data0/shaojiangyi/data/data-netgo/dataset_state_dict.pkl")
with open(dataset_path, "rb") as h:
    data = pickle.load(h)

In [16]:
prots_lst = [data["test"][x]["proteins"] for x in ontology_lst]

In [17]:
predprots_lst = [tp[0] for tp in methodpreds]

In [18]:
methodaligned = [
    align_labels(preds.T, o1, o2).T
    for preds, o1, o2 in zip(methodaligned, predprots_lst, prots_lst)
]

In [19]:
# bulid y_true
y_true = [data["test"][x]["prop_annotations"] for x in ontology_lst]
mlb_lst = [prep.MultiLabelBinarizer(classes=range(len(terms))) for terms in terms_lst]
targs = [mlb.fit_transform(y) for mlb, y in zip(mlb_lst, y_true)]

In [20]:
# def propagate(tp_tensor: np.ndarray | T.Tuple[np.ndarray, np.ndarray],
#               go_rels: gp.Ontology,
#               terms_df: pd.DataFrame,
#             #   top_k: int = 100,
#               min_score: float = 0.
#               ):
#     targs, preds = np.copy(tp_tensor)
#     assert isinstance(targs, np.ndarray) and isinstance(preds, np.ndarray)

#     # sorted_index = np.argsort(preds, 1)
#     # top_k_indices = sorted_index[:, :top_k]

#     mask = preds > min_score
#     rows = np.arange(preds.shape[0])[:, None]
#     # mask[rows, top_k_indices] = False

#     preds[~mask] = 0.

#     # propagate on the go graph
#     assert hasattr(terms_df, "gos")
#     term_idx = {k: i for i, k in enumerate(terms_df.gos)}
#     colmask = np.zeros(preds.shape[1], dtype=np.bool_)
#     for i in tqdm(np.arange(preds.shape[0]),total=preds.shape[0]):
#         pred_annots = terms_df.loc[mask[i], "gos"].tolist()
#         pred_scores = preds[i, mask[i]]
#         colmask.fill(False)
#         for go_id, score in zip(pred_annots, pred_scores):
#             supgo_set = go_rels.get_anchestors(go_id)
#             supgoids = [idx for t in supgo_set 
#                         if (idx := term_idx.get(t)) is not None]
#             colmask[supgoids] = True
#             preds[i, np.logical_and(colmask, ~mask[i])] = score
#             update_mask = np.logical_and(colmask, mask[i])
#             preds[i, update_mask] = np.maximum(preds[i, update_mask], score)
#     return targs, preds

In [21]:
def propagate_optimized(tp_tensor: np.ndarray | tuple[np.ndarray, np.ndarray],
                       go_rels,  # gp.Ontology
                       terms_df: pd.DataFrame,
                       min_score: float = 0.):
    """
    Optimized version of the propagate function with reduced time complexity.
    """
    targs, preds = np.copy(tp_tensor)
    assert isinstance(targs, np.ndarray) and isinstance(preds, np.ndarray)
    
    mask = preds > min_score
    preds[~mask] = 0.
    
    # Pre-compute term index mapping
    term_idx = {k: i for i, k in enumerate(terms_df.gos)}
    
    # Strategy 1: Pre-compute all ancestor mappings
    ancestor_cache = {}
    for go_id in terms_df.gos:
        supgo_set = go_rels.get_anchestors(go_id)
        ancestor_indices = [idx for t in supgo_set 
                          if (idx := term_idx.get(t)) is not None]
        ancestor_cache[go_id] = np.array(ancestor_indices, dtype=np.int32)
    
    # Process each sample
    for i in tqdm(range(preds.shape[0]), total=preds.shape[0]):
        sample_mask = mask[i]
        if not np.any(sample_mask):
            continue
            
        pred_annots = terms_df.loc[sample_mask, "gos"].tolist()
        pred_scores = preds[i, sample_mask]
        
        # Strategy 2: Batch process updates using vectorized operations
        for go_id, score in zip(pred_annots, pred_scores):
            ancestor_indices = ancestor_cache[go_id]
            
            # Update ancestors not already in mask (propagation)
            new_ancestors = ancestor_indices[~sample_mask[ancestor_indices]]
            preds[i, new_ancestors] = score
            
            # Update ancestors already in mask (take maximum)
            existing_ancestors = ancestor_indices[sample_mask[ancestor_indices]]
            preds[i, existing_ancestors] = np.maximum(preds[i, existing_ancestors], score)
    
    return targs, preds

In [22]:
# def propagate_sparse(tp_tensor: np.ndarray | T.Tuple[np.ndarray, np.ndarray],
#                     go_rels: gp.Ontology,
#                     terms_df: pd.DataFrame,
#                     min_score: float = 0.):
#     targs, preds = np.copy(tp_tensor)
    
#     mask = preds > min_score
#     preds[~mask] = 0.
    
#     # Build ancestor propagation matrix
#     term_idx = {k: i for i, k in enumerate(terms_df.gos)}
#     n_terms = len(terms_df)
    
#     # Create sparse matrix: rows=terms, cols=ancestors
#     rows, cols = [], []
#     for i, go_id in enumerate(terms_df.gos):
#         ancestors = go_rels.get_anchestors(go_id)
#         for anc in ancestors:
#             if anc in term_idx:
#                 rows.append(i)
#                 cols.append(term_idx[anc])
    
#     # Propagation matrix: if term i has ancestor j, then prop_matrix[i,j] = 1
#     prop_matrix = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), 
#                                shape=(n_terms, n_terms))
    
#     # Process all rows at once
#     for i in tqdm(range(preds.shape[0]), total=preds.shape[0]):
#         if not mask[i].any():
#             continue
            
#         # Get current predictions
#         current_preds = preds[i].copy()
        
#         # Propagate: for each term with prediction, add to all ancestors
#         propagated = prop_matrix.T @ current_preds  # ancestors get max of descendants
        
#         # Take maximum with existing predictions
#         preds[i] = np.maximum(preds[i], propagated)
    
#     return targs, preds

In [23]:
targ_dfs = [
    pd.DataFrame({"gos": terms})
    for terms in terms_lst
]

In [24]:
obo_path = P.Path("/data0/shaojiangyi/data/data-netgo/go.obo")
go_rels = gp.Ontology(obo_path, with_rels=True)

In [25]:
methodtp = [propagate_optimized((t, p), go_rels, terms_df)
            for t, p, terms_df in zip(targs, methodaligned, targ_dfs)]

  0%|          | 0/268 [00:00<?, ?it/s]

100%|██████████| 491/491 [01:17<00:00,  6.38it/s]


In [ ]:
# for t, p in methodtp:
#     assert p.shape == t.shape
#     print(hm.fmax_score(t, p, need_threshold=True, auprc=True))
""" result
((53.67138317065625, 1.0), 0.525048714668755)
((31.680198882756454, 0.8), 0.25325175339063594)
((26.68211987770605, 1.0), 0.25326884192646626)
"""

In [26]:
for t, p in methodtp:
    assert p.shape == t.shape
    print(hm.fmax_score(t, p, need_threshold=True, auprc=True))

((np.float64(0.5951310280164165), np.float64(0.4833034574985504)), np.float64(0.5774346762094831))
((np.float64(0.5987708515743964), np.float64(0.5557501316070557)), np.float64(0.5986647402591152))
((np.float64(0.32723190615264325), np.float64(0.48437824845314026)), np.float64(0.22215614933078065))


In [31]:
# original no propagation
for t, p in zip(targs, methodaligned):
    assert p.shape == t.shape
    (f, t), a = hm.fmax_score(t, p, need_threshold=True, auprc=True)
    print(f"({f}, {t}), {a}")

(0.5842066420164371, 0.43592551350593567), 0.5736424219674601
(0.5996384368850589, 0.5193024277687073), 0.6008731372011693
(0.33322663247261414, 0.45653167366981506), 0.21974917324007712


In [32]:
# original no propagation
for t, p in zip(targs, methodaligned):
    assert p.shape == t.shape
    (f, t), a = hm.fmax_score(t, p, need_threshold=True, auprc=True, no_empty_labels=True, no_zero_classes=True)
    print(f"({f}, {t}), {a}")

(0.6030778606844294, 0.43592551350593567), 0.6024357675123889
(0.6212754106990969, 0.5034577250480652), 0.6346092066495039
(0.35270423902760034, 0.4448246955871582), 0.2478718363339692


In [33]:
# saving
saving_path = [methodpred_path[i].parent / f"{x}_prop_pred_result.npy" for i, x in enumerate(ns)]
for path, tp in zip(saving_path, methodtp):
    np.save(path, np.stack(tp, axis=0))

In [34]:
saving_path = [methodpred_path[i].parent / f"{x}_pred_result.npy" for i, x in enumerate(ns)]
for path, t, p in zip(saving_path, targs, methodaligned):
    np.save(path, np.stack((t, p), axis=0))